Inspect the data inside the truthtriplet files.

These are the upstream data files we will make the prong embed training data from.

To make the file that goes into this notebook run `run_lardata2hdf5.py`

Script found in `ubdl/larflow/larmatchnet/larmatch/`
```
python3 run_lardata2hdf5.py --input-larlite [larlite_file.root] --input-larcv [larcv_file.root] -o [outfile.h5] -tb -tri
```

In [1]:
import chart_studio as cs
import chart_studio.plotly as py
import plotly.graph_objects as go
import dash
from dash import Dash, html, dcc, Input, Output, callback
from dash.exceptions import PreventUpdate
from ctypes import c_int
import numpy as np
from larmatch.data.larmatch_hdf5_reader import LArMatchHDF5Dataset, get_data_loader
%load_ext autoreload
%autoreload 2

/home/twongjirad/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# data useful for plots
import lardly
from lardly import DetectorOutline # load utility to draw TPC outline
detdata = DetectorOutline()
detlines = detdata.getlines(color=(10,10,10))

# PARTICLE LABEL COLORS
ssnetcolor = {-1:np.array((0,0,0)),     # ghost                                                                                                                                                   
              0:np.array((255,0,0)),   # electron                                                                                                                                       
              1:np.array((0,255,0)),   # gamma                                                                                                                             
              2:np.array((0,0,255)),   # muon                                                                                                                                              
              3:np.array((255,0,255)), # proton                                                                                                                                                 
              4:np.array((0,255,255)) # pion (+other mesons)
             }


ssnetnames = {-1:"ghost",
             0:"e",
             1:"gamma",
             2:"mu",
             3:"proton",
             4:"pion"}

kpcolors = {0:np.array((255,0,0)), # nu (red)
            1:np.array((0,255,0)), # track-start (green)
            2:np.array((0,0,255)), # track-end (blue)
            3:np.array((255,128,0)), # shower (orange)
            4:np.array((0,255,255)), # michel
            5:np.array((255,0,255))} # delta
kpnames = {0:"neutrino",
          1:"track-start",
          2:"track-end",
          3:"shower-start",
          4:"michel-start",
          5:"delta-start"}


Welcome to JupyROOT 6.24/02


In [3]:
inputfiles = ["larmatch_trainingdata-Run000001-SubRun000002.h5"]
#inputfiles = ["larmatch_trainingdata_536038de-1412-4db2-bca3-acaaaa0a564c.h5"]
#inputfiles = ["mcc9_v40_run3b_NCpi0_overlay_aa444faa-530a-4fd7-b43f-b501bc221880.h5"]

In [4]:
dataset = get_data_loader( inputfiles, batch_size=1, shuffle=False, num_workers=1, 
                          collate_for_training=True, apply_max_filter=False)
iter_data = iter(dataset)

Loading data files from list of file paths
Make entry table for list of files (len= 1 )
MAKE_ENTRY_TABLE: Loading from list of file paths
length= 20  for  larmatch_trainingdata-Run000001-SubRun000002.h5


In [5]:
data = next(iter_data)

In [6]:
batch_size = len(data)
print(data[0].keys())

ibatch = 0
batchdata = data[ibatch]

dict_keys(['matchtriplet_v', 'larmatch_truth', 'larmatch_weight', 'ssnet_truth', 'ssnet_weight', 'keypoint_truth', 'keypoint_weight', 'paf_label', 'paf_weight', 'spacepoints', 'keypoint_truth_pos', 'keypoint_truth_kptype_pdg_trackid', 'coord_0', 'feat_0', 'query_coord_0', 'coord_1', 'feat_1', 'query_coord_1', 'coord_2', 'feat_2', 'query_coord_2', 'idx'])


In [7]:
print(batchdata['spacepoints'].shape)
print(batchdata['matchtriplet_v'].shape)

(597217, 3)
(597217, 5)


In [ ]:
from larmatch.data.samplers import larmatch_example_balancer

batchdata = larmatch_example_balancer(batchdata, max_nspacepoints_returned=125000)


In [8]:
# extract the true keypoint locations
kppos  = batchdata['keypoint_truth_pos']
kptype = batchdata['keypoint_truth_kptype_pdg_trackid'][:,0]
kppdg  = batchdata['keypoint_truth_kptype_pdg_trackid'][:,1]
kptid  = batchdata['keypoint_truth_kptype_pdg_trackid'][:,2]
keypoint_plots = []
for ikptype in kpnames:
    mask = kptype==ikptype
    pos = kppos[ mask ]
    xpdg = kppdg[mask]
    xtid = kptid[mask]
    print("Number of %s keypoints: %d"%(kpnames[ikptype],pos.shape[0]))
    hovertemplate='<b>PDG</b>: %{customdata[0]}<br>' + \
                  '<b>Tid</b>: %{customdata[1]}<br>' + \
                  'x: %{x}<br>' + \
                  'y: %{y}<br>' + \
                  'z: %{z}<br>'
    kptype_plot = {
    "type":"scatter3d",
    "x": pos[:,0],
    "y": pos[:,1],
    "z": pos[:,2],
    "mode":"markers",
    "name":kpnames[ikptype],
    "customdata":np.stack( (xpdg,xtid), axis=-1 ),
    "marker":{"color":"rgb(%d,%d,%d)"%tuple(kpcolors[ikptype]),"size":5.0,"opacity":0.5},
    "hovertemplate":hovertemplate
    }
    if kpnames[ikptype]=="neutrino":
        kptype_plot['marker']['size'] = 10.0
    keypoint_plots.append( kptype_plot )


Number of neutrino keypoints: 1
Number of track-start keypoints: 19
Number of track-end keypoints: 19
Number of shower-start keypoints: 1
Number of michel-start keypoints: 1
Number of delta-start keypoints: 28


In [ ]:
# Prepare spacepoint data and figure
# Plot the true/ghost labels

# use a limit to speed up plotting (will randomly sample)
LIMIT_NUM_SPACEPOINTS=True
MAX_NUM_SAMPLES=300000
TRUE_POINTS_ONLY=False

# get the arrays
triplets   = batchdata['matchtriplet_v']
print('triplets.shape: ',triplets.shape)
spacepoints = batchdata['spacepoints']
print("spacepoints.shape: ",spacepoints.shape)
truthlabels = triplets[:,3]


if TRUE_POINTS_ONLY:
    xmask = triplets[:,3]==1
    xspacepoints = spacepoints[xmask]
    xtriplets    = triplets[xmask]
else:
    xspacepoints = spacepoints
    xtriplets = triplets

npts = xspacepoints.shape[0]
xtruthlabel  = xtriplets[:npts,3]
print("triplets.shape: ",triplets.shape)
wireimgs = [ batchdata['coord_%d'%(i)] for i in range(3) ]

if LIMIT_NUM_SPACEPOINTS:
    xrandom = np.random.random(npts)<float(MAX_NUM_SAMPLES)/float(npts)
    xpos = xspacepoints[xrandom[:],:]
    xtruth = xtruthlabel[xrandom[:]]
    xtriplets = xtriplets[xrandom[:],:3]
else:
    xpos = spacepoints
    xtruth = truthlabel
    xtriplets = triplets[:,:3]
#print(spacepoints[:10])
#print(truthlabel[:10])

def make_callback_customdata(wireimgs,xtriplets):
    xucol = wireimgs[0][xtriplets[:,0],1]
    xurow = wireimgs[0][xtriplets[:,0],0]
    xvcol = wireimgs[1][xtriplets[:,1],1]
    xvrow = wireimgs[1][xtriplets[:,1],0]
    xycol = wireimgs[2][xtriplets[:,2],1]
    xyrow = wireimgs[2][xtriplets[:,2],0]
    customdata = np.stack( (xucol,xurow,xvcol,xvrow,xycol,xyrow), axis=-1)
    return customdata

spacepoint_customdata = make_callback_customdata(wireimgs,xtriplets)

print("custom.shape: ",spacepoint_customdata.shape)
hovertemplate='<b>U</b>: (%{customdata[0]},%{customdata[1]})<br>' + \
            '<b>V</b>: (%{customdata[2]},%{customdata[3]})<br>' + \
            '<b>Y</b>: (%{customdata[4]},%{customdata[5]})<br>' + \
            'x: %{x}<br>' + \
            'y: %{y}<br>' + \
            'z: %{z}<br>'

# define a 3d scatter plot in plotly
plot_spacepoints = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"spacepoints",
    "customdata":spacepoint_customdata,
    "hovertemplate":hovertemplate,
    "marker":{"color":xtruth.astype(np.float64),"size":1.0,"opacity":0.5,"colorscale":"Bluered"},
    }


# collect the things we want to plot
# we add the spacepoint plot to the outline of the uboone detector
spacepoint_plot_list = detlines + [plot_spacepoints]
try:
    spacepoint_plot_list += keypoint_plots
except:
    print("no keypoint_plots?")

# define axes and layout of plot
axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}
layout_spacepoint = go.Layout(
    title='Truth/Ghost Labels',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)


# make the plot using plotly go
spacepoint_fig = go.Figure(data=spacepoint_plot_list, layout=layout_spacepoint)
#fig.show()

In [ ]:
# prepare wire plane image data
## Wire plane images

wireimgs = [ batchdata['coord_%d'%(i)] for i in range(3) ]
wireplots = []
for iplane,img in enumerate(wireimgs):
    pixval = batchdata['feat_%d'%(iplane)]
    plot = {
        "type":"scatter",
        "mode":"markers",
        "x":img[:,1],#wires
        "y":img[:,0],#ticks
        "marker":{"color":pixval,"colorscale":"Viridis","symbol":"square","size":4,"opacity":0.5}
    }
    wireplots.append(plot)

    
# define axes and layout of plot
wireimage_axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}


# make the plot using plotly go
wireimage_figures = []
for p in range(3):
    wireimage_layout = go.Layout(
        title='wire plane %d'%(p),
        autosize=True,
        hovermode='closest',
        showlegend=False,
    )
    wireimage_figures.append( go.Figure(data=[wireplots[p]], layout=wireimage_layout) )
    #wireimage_figures[-1].show()

In [ ]:
# PREPARE SSNET DATA

ssnetdata   = batchdata['ssnet_truth']
spacepoints = batchdata['spacepoints']
triplets    = batchdata['matchtriplet_v']
print("ssnetdata.shape: ",ssnetdata.shape)

SSNET_LIMIT_NUM_SPACEPOINTS=True
SSNET_MAX_NUM_SAMPLES=300000

# Flag set above
if TRUE_POINTS_ONLY:
    xmask = triplets[:,3]==1
    xspacepoints = spacepoints[xmask]
    xtriplets    = triplets[xmask]
    xssnetdata   = ssnetdata[xmask]
else:
    xspacepoints = spacepoints
    xtriplets    = triplets
    xssnetdata   = ssnetdata
    
if LIMIT_NUM_SPACEPOINTS and xssnetdata.shape[0]>MAX_NUM_SAMPLES:
    # use filter from before (xrandom)
    xspacepoints = xspacepoints[xrandom]
    xssnetdata   = xssnetdata[xrandom]
    xtriplets    = xtriplets[xrandom]

ssnet_plots = []

# first isolate by class
for iclass in ssnetcolor:
    xmask = xssnetdata==iclass
    xlabels = xssnetdata[xmask]
    xpos = xspacepoints[xmask[:],:]
    xssnettriplet = xtriplets[xmask[:],:]
    # downsample if needed
#     if SSNET_LIMIT_NUM_SPACEPOINTS and xlabels.shape[0]>SSNET_MAX_NUM_SAMPLES:
#         factor = float(SSNET_MAX_NUM_SAMPLES)/float(xlabels.shape[0])
#         xfilter = np.random.random( xlabels.shape[0] ) < factor
#         xlabels = xlabels[ xfilter ]
#         xpos = xpos[ xfilter[:], :]
    xssnet_customdata_iclass = make_callback_customdata( wireimgs, xssnettriplet )
    
    print("number of ssnet class [",ssnetnames[iclass],"] spacepoints: ",xpos.shape[0])
    if xpos.shape[0]==0:
        continue
    ptsize = 1.0
    if iclass==0:
        # reduce ghost point size
        ptsize = 0.5
        
    ssnet_plot = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "customdata":xssnet_customdata_iclass,
    "hovertemplate":hovertemplate,
    "name":"[%d] %s"%(iclass,ssnetnames[iclass]),
    "marker":{"color":"rgb(%d,%d,%d)"%tuple(ssnetcolor[iclass]),"size":ptsize,"opacity":0.5},
    }
    ssnet_plots.append( ssnet_plot )

ssnet_plot_traces = detlines + ssnet_plots
try:
    ssnet_plot_traces += keypoint_plots
except:
    print("no keypoint plots")

layout = go.Layout(
    title='SSNet Labeled Spacepoints',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

ssnet_fig = go.Figure(data=ssnet_plot_traces, layout=layout)

In [ ]:
# DASH Application
app = Dash(__name__)

# define some defaults
def make_imagecrop_figure( col, row, img, imgfeat, planeid ):
    ucolfilter= (img[:,1]>=col-20) * (img[:,1]<=col+20)
    urowfilter= (img[:,0]>=row-20) * (img[:,0]<=row+20)
    ufilter = ucolfilter * urowfilter
    #print(ufilter.sum(),ufilter.shape)
    xuimg_coords = img[ufilter[:],:2].astype(np.int32)
    xuimg_feat   = imgfeat[ufilter[:]].squeeze()
    #xuimg_feat.reshape( (xuimg_feat.shape[0],1) )
    #print(xuimg.shape)
    
    col_offset = col-20
    row_offset = row-20
    z = np.zeros((41,41))
    z[ xuimg_coords[:,0]-row_offset,xuimg_coords[:,1]-col_offset ] = xuimg_feat
    x = np.arange(col_offset,col_offset+41)
    y = np.arange(row_offset,row_offset+41)
    uplot = go.Heatmap(z=z,x=x,y=y)
    return uplot

# default images for wireplane crops
empty_heatmap_v = []
for p in range(3):
    layout_p = go.Layout(
        title='wire plane %d'%(p),
        width=300, 
        height=300,
        #autosize=True,
        hovermode='closest',
        showlegend=False)
    empty_heatmap_v.append( go.Figure( data=[go.Heatmap(z=np.zeros((41,41)))], layout=layout_p ) )

# ====================
# SET THE 3D PLOT
# ====================
#point_plot = spacepoint_fig
point_plot = ssnet_fig
#point_plot = 
    

app.layout = html.Div([
    # top row is the spacepoint graph
    html.Div(dcc.Graph(figure=point_plot, id='spacepoint-graph'),
            style={'width':'99%','display':'inline-block','padding': '0px 0px 0px 0px'}),
    # right-hand stack of wireplanes
    html.Div([
        # uplane
        html.Div(
            dcc.Graph(figure=empty_heatmap_v[0], id='plane0-graph'), 
            style={'display':'inline-block','padding': '0px 0px 0px 0px','width':'32%'}),
        # vplane
        html.Div(
            dcc.Graph(figure=empty_heatmap_v[1], id='plane1-graph'),
            style={'display':'inline-block','padding': '0px 0px 0px 0px','width':'32%'}),
        # yplane
        html.Div(
            dcc.Graph(figure=empty_heatmap_v[2], id='plane2-graph'),
            style={'display':'inline-block','padding': '0px 0px 0px 0px','width':'32%'})],
        style={'width': '99%', 'display': 'inline-block','float':'right'})
])#end of wire image rows


@callback(
    Output('plane0-graph', 'figure'),
    Output('plane1-graph', 'figure'),
    Output('plane2-graph', 'figure'),
    Input('spacepoint-graph', 'hoverData'))
def update_wireplane_crops(hoverData):
    #x = hoverData['points'][0]
    #print(x)
    plots = {0:None,
             1:None,
             2:None}
    updated = False
    if hoverData:
        #print("hello: ",hoverData.keys())
        x = hoverData['points'][0]
        if 'customdata' in x and len(x['customdata'])==6:
            #print('customdata: ',x['customdata'])
            # we crop the data
            for p in range(3):
                uwire = x['customdata'][2*p]
                urow  = x['customdata'][2*p+1]
                uimg = batchdata['coord_%d'%(p)]
                upix = batchdata['feat_%d'%(p)]
                wireimage_layout = go.Layout(
                    title='wire plane %d'%(p),
                    #autosize=True,
                    hovermode='closest',
                    showlegend=False)
                xx = make_imagecrop_figure( uwire, urow, uimg, upix, p )
                plots[p] = go.Figure(data=[xx],layout=wireimage_layout)
                updated = True

    if updated:
        return plots[0],plots[1],plots[2]
    else:
        return dash.no_update

In [ ]:
if __name__=="__main__":
    app.run()

In [ ]:
# Plot the true/ghost labels

# use a limit to speed up plotting (will randomly sample)
LIMIT_NUM_SPACEPOINTS=True
MAX_NUM_SAMPLES=50000

import torch
import pytorch3d
import pytorch3d.ops as ops
#ops.sample_farthest_points
# """
# def sample_farthest_points(
#     points: torch.Tensor,
#     lengths: Optional[torch.Tensor] = None,
#     K: Union[int, List, torch.Tensor] = 50,
#     random_start_point: bool = False,
# ) -> Tuple[torch.Tensor, torch.Tensor]:
#     """
#     Iterative farthest point sampling algorithm [1] to subsample a set of
#     K points from a given pointcloud. At each iteration, a point is selected
#     which has the largest nearest neighbor distance to any of the
#     already selected points.

#     Farthest point sampling provides more uniform coverage of the input
#     point cloud compared to uniform random sampling.
# """

# get the arrays
triplets   = batchdata['matchtriplet_v']
print('triplets.shape: ',triplets.shape)
spacepoints = torch.from_numpy( batchdata['spacepoints'] )
N = spacepoints.shape[0]
print("spacepoints.shape: ",spacepoints.shape)
npts = spacepoints.shape[0]
truthlabel  = triplets[:npts,3]
print("triplets.shape: ",triplets.shape)
#wireimgs = [ batchdata['wireimage_plane%d'%(i)] for i in range(3) ]

bspacepoints = spacepoints.unsqueeze(0) 
print("bspacepoints=",bspacepoints.shape)
selected_points, selected_indices = ops.sample_farthest_points( bspacepoints, K=int(npts/10))
print("Furthest point sampling: ",selected_points.shape )
print(selected_points)

if LIMIT_NUM_SPACEPOINTS:
    xrandom = np.random.random(npts)<float(MAX_NUM_SAMPLES)/float(npts)
    xpos = spacepoints[xrandom[:],:]
    xtruth = truthlabel[xrandom[:]]
    xtriplets = triplets[xrandom[:],:3]
else:
    xpos = spacepoints
    xtruth = truthlabel
    xtriplets = triplets[:,:3]
#print(spacepoints[:10])
#print(truthlabel[:10])

xucol = wireimgs[0][xtriplets[:,0],1]
xurow = wireimgs[0][xtriplets[:,0],0]
xvcol = wireimgs[1][xtriplets[:,1],1]
xvrow = wireimgs[1][xtriplets[:,1],0]
xycol = wireimgs[2][xtriplets[:,2],1]
xyrow = wireimgs[2][xtriplets[:,2],0]
customdata = np.stack( (xucol,xurow,xvcol,xvrow,xycol,xyrow), axis=-1)


print("custom.shape: ",customdata.shape)
hovertemplate='<b>U</b>: (%{customdata[0]},%{customdata[1]})<br>' + \
            '<b>V</b>: (%{customdata[2]},%{customdata[3]})<br>' + \
            '<b>Y</b>: (%{customdata[4]},%{customdata[5]})<br>' + \
            'x: %{x}<br>' + \
            'y: %{y}<br>' + \
            'z: %{z}<br>'

# define a 3d scatter plot in plotly
plot_spacepoints = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"spacepoints",
    "customdata":customdata,
    "hovertemplate":hovertemplate,
    "marker":{"color":xtruth.astype(np.float64),"size":1.0,"opacity":0.5,"colorscale":"Bluered"},
    }

plot_centroids = {
    "type":"scatter3d",
    "x":selected_points[0,:,0],
    "y":selected_points[0,:,1],
    "z":selected_points[0,:,2],
    "mode":"markers",
    "name":"spacepoints",
    "marker":{"color":"rgba(255,255,0,1)","size":3.0,"opacity":0.5},
    }

# collect the things we want to plot
# we add the spacepoint plot to the outline of the uboone detector
spacepoint_plot_list = detlines + [plot_spacepoints,plot_centroids]
#try:
#    spacepoint_plot_list += keypoint_plots
#except:
#    print("no keypoint_plots?")

# define axes and layout of plot
axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}
layout_spacepoint = go.Layout(
    title='Truth/Ghost Labels',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)


# make the plot using plotly go
spacepoint_fig = go.Figure(data=spacepoint_plot_list, layout=layout_spacepoint)
spacepoint_fig.show()

In [ ]:
# PLOT WEIGHTS
try:
    del plot_weights
    del weight_fig
    del plotvalues
except:
    pass

# get the arrays
triplets   = batchdata['matchtriplet_v']
print('triplets.shape: ',triplets.shape)
spacepoints = batchdata['spacepoints']
print("spacepoints.shape: ",spacepoints.shape)
npts = spacepoints.shape[0]
truthlabel  = triplets[:npts,3]
print("triplets.shape: ",triplets.shape)
print('keypoint_weight.shape: ',batchdata['keypoint_weight'].shape)

# PLOT LARMATCH WEIGHTS
#plotvalues = batchdata['larmatch_weight'][:,0]*float(npts)

# PLOT KEYPOINT CLASS WEIGHTS
#iplot_kpclass = 0
#plotvalues = 8.0+np.log10( 1.001e-8 + batchdata['keypoint_weight'][iplot_kpclass,:] )

# SSNET WEIGHTS
#vals = batchdata['ssnet_weight']
#plotvalues = 8.0+np.log10( 1.001e-8 + batchdata['ssnet_weight'] )

# PAF WEIGHTS
vals = batchdata['paf_weight']
plotvalues = 8.0+np.log10( 1.001e-8 + vals )

print('plotvalues.shape: ',plotvalues.shape)
maxval = plotvalues.max()
print("max weight value: ",plotvalues.max())
print("min weight value: ",plotvalues.min())
vals = np.unique(vals)
for i,v in enumerate(vals):
    if v>0:
        print("[",i,"] weight val=",v," implied num samples 1/val=",1/v)
    else:
        print("[",i,"] weight val=",v)
print("unique log(weight) values: ",np.unique(plotvalues))
xpos = spacepoints

print("x range: ",xpos[:,0].min()," ",xpos[:,0].max())
print("y range: ",xpos[:,1].min()," ",xpos[:,1].max())
print("z range: ",xpos[:,2].min()," ",xpos[:,2].max())

# define a 3d scatter plot in plotly
plot_weights = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"weights",
    "marker":{"color":plotvalues,"size":1.0,"opacity":0.5,"colorscale":"Viridis","cmin":0.0,"cmax":maxval},
    }


# collect the things we want to plot
# we add the spacepoint plot to the outline of the uboone detector
weights_plot_list = detlines + [plot_weights]
#try:
#    weights_plot_list += keypoint_plots
#except:
#    print("no keypoint_plots?")

# define axes and layout of plot
axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}
layout_weights = go.Layout(
    title='Spacepoint Weights',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)


# make the plot using plotly go
weight_fig = go.Figure(data=weights_plot_list, layout=layout_weights)
weight_fig.show()

In [ ]:
# PREPARE SSNET DATA

ssnetdata = batchdata['ssnet_label']
spacepoints = batchdata['spacepoints']
print("ssnetdata.shape: ",ssnetdata.shape)

SSNET_LIMIT_NUM_SPACEPOINTS=True
SSNET_MAX_NUM_SAMPLES=10000

ssnet_plots = []

# first isolate by class
for iclass in ssnetcolor:
    xmask = ssnetdata==iclass
    xlabels = ssnetdata[xmask]
    xpos = spacepoints[xmask[:],:]
    # downsample if needed
    if SSNET_LIMIT_NUM_SPACEPOINTS and xlabels.shape[0]>SSNET_MAX_NUM_SAMPLES:
        factor = float(SSNET_MAX_NUM_SAMPLES)/float(xlabels.shape[0])
        xfilter = np.random.random( xlabels.shape[0] ) < factor
        xlabels = xlabels[ xfilter ]
        xpos = xpos[ xfilter[:], :]
    
    print("number of ssnet class [",ssnetnames[iclass],"] spacepoints: ",xpos.shape[0])
    if xpos.shape[0]==0:
        continue
    ptsize = 1.0
    if iclass==0:
        # reduce ghost point size
        ptsize = 0.5
        
    ssnet_plot = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"[%d] %s"%(iclass,ssnetnames[iclass]),
    "marker":{"color":"rgb(%d,%d,%d)"%tuple(ssnetcolor[iclass]),"size":ptsize,"opacity":0.5},
    }
    ssnet_plots.append( ssnet_plot )

ssnet_plot_traces = detlines + ssnet_plots
try:
    ssnet_plot_traces += keypoint_plots
except:
    print("no keypoint plots")

layout = go.Layout(
    title='SSNet Labeled Spacepoints',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

fig = go.Figure(data=ssnet_plot_traces, layout=layout)
fig.show()


In [ ]:
# Plot by origin label

origin = batchdata['origin_label']
spacepoints = batchdata['spacepoints']
print(np.unique(origin))
print("origin.shape: ",origin.shape)

origin_names = {0:'ghost',
               1:'neutrino',
               2:'cosmic'}
origin_colors = {0:'rgb(0,0,0)',
                1:'rgb(255,0,0)',
                2:'rgb(0,0,255)'}

ORIGIN_LIMIT_NUM_SPACEPOINTS=True
ORIGIN_MAX_NUM_SAMPLES=10000

origin_plots = []

for iorigin in [0,1,2]:
    xmask = origin==iorigin
    xpos = spacepoints[xmask[:],:]
    if ORIGIN_LIMIT_NUM_SPACEPOINTS and xpos.shape[0]>ORIGIN_MAX_NUM_SAMPLES:
        factor = float(ORIGIN_MAX_NUM_SAMPLES)/float(xpos.shape[0])
        xfilter = np.random.random( xpos.shape[0] ) < factor
        xpos = xpos[xfilter[:],:]
        
    ptsize = 1.0
    if iorigin==0:
        ptsize = 0.5
    plot = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"[%d] %s"%(iorigin,origin_names[iorigin]),
    "marker":{"color":origin_colors[iorigin],"size":ptsize,"opacity":0.5},
    }
    origin_plots.append( plot )

origin_plot_traces = detlines + origin_plots
try:
    origin_plot_traces += keypoint_plots
except:
    print("no keypoint plots")

layout = go.Layout(
    title='Origin labeled Spacepoints',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

fig = go.Figure(data=origin_plot_traces, layout=layout)
fig.show()


In [ ]:
# Direction labels

pf = batchdata['paf_label'].squeeze()
pfw = batchdata['paf_weight']
spacepoints = batchdata['spacepoints']
truemask = batchdata['matchtriplet_v'][:,3]==1
truemask *= pfw>0.0
pftrue = pf[:,truemask[:]]
sptrue = spacepoints[truemask[:],:]

print('pftrue.shape: ',pftrue.shape)

PF_LIMIT_NUM_SPACEPOINTS=True
PF_MAX_NUM_SAMPLES=10000

if PF_LIMIT_NUM_SPACEPOINTS and pftrue.shape[1]>PF_MAX_NUM_SAMPLES:
    factor = float(PF_MAX_NUM_SAMPLES)/float(pftrue.shape[1])
    xfilter = np.random.random( pftrue.shape[1] ) < factor
    print("xfilter sum: ",xfilter.sum())
    xpftrue = pftrue[:,xfilter[:]]
    xsptrue = sptrue[xfilter[:],:]
else:
    xpftrue = pftrue
    xsptrue = sptrue

print("spacepoint list.",xsptrue.shape)
print(xsptrue[:10,])
print("xpftrue list. shape=",xpftrue.shape)
print(xpftrue[:,:10])

print("num paf spacepoints: ",xpftrue.shape)

N=xsptrue.shape[0]
#N=100

pf_plot = {
    "type":"cone",
    "x": xsptrue[:N,0],
    "y": xsptrue[:N,1],
    "z": xsptrue[:N,2],
    "u": xpftrue[0,:N]*1.0,
    "v": xpftrue[1,:N]*1.0,
    "w": xpftrue[2,:N]*1.0,
    "name":"pflow",
    "sizemode":"absolute",
    "sizeref":20,
    "anchor":"tail"
    }

pf_traces = detlines + [pf_plot]

layout = go.Layout(
    title='Particle Flow Direction',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

fig = go.Figure(data=pf_traces, layout=layout)
fig.show()

In [ ]:
## Wire plane images

wireimgs = [ batchdata['wireimage_plane%d'%(i)] for i in range(3) ]
wireplots = []
for iplane,img in enumerate(wireimgs):
    plot = {
        "type":"scatter",
        "mode":"markers",
        "x":img[:,1],#wires
        "y":img[:,0],#ticks
        "marker":{"color":img[:,2],"colorscale":"Viridis","symbol":"square","size":4,"opacity":0.5}
    }
    wireplots.append(plot)

    
# define axes and layout of plot
wireimage_axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}


# make the plot using plotly go
wireimage_figures = []
for p in range(3):
    wireimage_layout = go.Layout(
        title='wire plane %d'%(p),
        autosize=True,
        hovermode='closest',
        showlegend=False,
    )
    wireimage_figures.append( go.Figure(data=[wireplots[p]], layout=wireimage_layout) )
    wireimage_figures[-1].show()